![image.png](https://i.imgur.com/a3uAqnb.png)

# Speculative Decoding with Medusa Draft Heads

- **Dataset**: [UltraChat 200k](https://huggingface.co/datasets/HuggingFaceH4/ultrachat_200k) from Hugging Face, about 208,000 multi-turn assistant dialogues (the `train_sft` split), MIT licence
- **Task**: train draft heads that guess the tokens 2, 3 and 4 positions ahead, then use them to decode several tokens per forward pass of the target model
- **Model**: [Qwen/Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) as a frozen target, with three Medusa heads (about 700M parameters) trained from scratch on its final hidden state. [Qwen/Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) serves as a separate draft model for the comparison at the end
- **Runtime**: a Colab A100 GPU with at least 40 GB of memory, about 25 minutes of compute of which about 12 minutes are training. The heads and their optimizer state do not fit a free T4

In this lab, we will:

- tokenise real assistant conversations with the model's chat template
- build Medusa draft heads as residual blocks on the frozen final hidden state and train them with a per-head cross-entropy
- implement the draft-and-verify decoding loop with greedy verification and count accepted tokens
- measure acceptance length and wall-clock speedup against plain decoding
- compare our heads with the built-in assisted generation of `transformers`, which uses a separate draft model

This notebook is adapted from the [Medusa repository](https://github.com/FasterDecoding/Medusa) (Cai et al., 2024) and the Hugging Face post [Assisted Generation: a new direction toward low-latency text generation](https://huggingface.co/blog/assisted-generation) (Gante, 2023).

## What is speculative decoding?

Decoding one token per forward pass leaves a GPU idle: every step reads all the weights to produce one row of logits. Speculative decoding fills that idle compute. A cheap *draft* proposes $K$ tokens and the large *target* checks all of them in one forward pass, which costs about as much as generating a single token. Under greedy decoding a drafted token $\tilde{x}$ is accepted exactly when it equals the target's own choice, so the output is identical to plain decoding. Under sampling, the rule that keeps the target's distribution is

$$\Pr[\text{accept } \tilde{x}] = \min\left(1,\ \frac{q(\tilde{x})}{p(\tilde{x})}\right)$$

Where:
- $p$ is the draft's distribution over the next token and $q$ is the target's
- at the first rejection the token is resampled from the normalised positive part of $q - p$ and the rest of the draft is discarded
- if a drafted token is accepted with probability $\alpha$ on average, one target pass yields $(1 - \alpha^{K+1}) / (1 - \alpha)$ tokens in expectation

Medusa removes the separate draft model. It adds $K$ small *heads* to the target, all reading the final hidden state $h_t$ that already exists after the target's forward pass, and head $k$ guesses the token at position $t + k + 1$:

$$p_t^{(k)} = \mathrm{softmax}\left(W_2^{(k)}\left(\mathrm{SiLU}(W_1^{(k)} h_t) + h_t\right)\right), \qquad \mathcal{L} = \sum_{k=1}^{K} -\lambda_k \log p_t^{(k)}(y_{t+k+1})$$

Where:
- $W_1^{(k)} \in \mathbb{R}^{d \times d}$ starts at zero and $W_2^{(k)} \in \mathbb{R}^{V \times d}$ starts as a copy of the language-model head, so every head begins as the original next-token head
- $\lambda_k = 0.8^k$ gives less weight to the harder far-ahead heads, and the backbone stays frozen

Papers: [Leviathan et al., 2023. Fast Inference from Transformers via Speculative Decoding](https://arxiv.org/abs/2211.17192), [Chen et al., 2023. Accelerating Large Language Model Decoding with Speculative Sampling](https://arxiv.org/abs/2302.01318), [Cai et al., 2024. Medusa: Simple LLM Inference Acceleration Framework with Multiple Decoding Heads](https://arxiv.org/abs/2401.10774)

## Setup

This lab needs an A100 GPU with at least 40 GB of memory: in Google Colab, select **Runtime > Change runtime type > A100 GPU**, then run the cells from top to bottom. Colab already ships `transformers`, `datasets` and `accelerate`, so nothing is installed. Each head holds a full vocabulary projection (about 235M parameters), and with their AdamW state the three heads need about 12 GB before activations, so a T4 runs out of memory.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from datasets import load_dataset
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache, get_cosine_schedule_with_warmup

In [ ]:
SEED = 42
TARGET_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"     # frozen backbone whose final hidden state the heads read
DRAFT_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"      # separate draft model, only for the comparison at the end
DATASET = "HuggingFaceH4/ultrachat_200k"
TRAIN_FILE = "data/train_sft-00000-of-00003-a3ecf92756993583.parquet"   # one of three train_sft shards, 244 MB
TEST_FILE = "data/test_sft-00000-of-00001-f7dfac4afe5b93f4.parquet"     # the test_sft split, 81 MB
N_TRAIN = 5_000          # conversations for training (the shard holds about 69,000). With 2,000 the heads
                         # overfit in the second epoch: the training loss kept falling while the held-out loss rose
N_TEST = 200             # held-out conversations
MAX_LEN = 1024           # tokens kept per conversation
BATCH_SIZE = 8
EPOCHS = 2
LR = 1e-3                # Medusa trains the heads with a large rate because the backbone is frozen
WARMUP_STEPS = 40        # linear warm-up, then cosine decay to zero, as in the Medusa training script
N_HEADS = 3              # head k guesses the token k + 1 positions ahead
HEAD_LAYERS = 1          # residual blocks per head
LOSS_DECAY = 0.8         # lambda_k = LOSS_DECAY ** k weights the loss of head k
NUM_WORKERS = 2          # DataLoader worker processes, a Colab runtime has 2 CPU cores
N_EVAL_PROMPTS = 20      # prompts for the decoding benchmark
MAX_NEW_TOKENS = 128     # tokens generated per prompt in the benchmark

torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
else:
    print("No GPU found, so the model will not fit and training will be very slow. "
          "In Colab: Runtime > Change runtime type > A100 GPU.")

# Mixed precision: bfloat16 on an A100 or newer (compute capability 8), float16 with a gradient scaler
# on an older GPU, off on the CPU. The frozen backbone is loaded directly in that dtype.
USE_AMP = DEVICE == "cuda"
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE == torch.float16)
MODEL_DTYPE = AMP_DTYPE if USE_AMP else torch.float32
if USE_AMP and AMP_DTYPE == torch.bfloat16:
    torch.set_float32_matmul_precision("high")      # TF32 for the float32 head matmuls outside autocast

## 1. Dataset

- **Source**: [UltraChat 200k](https://huggingface.co/datasets/HuggingFaceH4/ultrachat_200k) on Hugging Face (Ding et al., 2023), the filtered dialogues used to train Zephyr-7B
- **Content**: 207,865 training and 23,110 test dialogues between a user and an assistant (`train_sft` and `test_sft`), each a list of `messages` with `role` and `content`, stored as parquet
- **Licence**: MIT
- **What we use**: 5,000 training dialogues from one of the three `train_sft` shards and 200 test dialogues, truncated to 1,024 tokens, about 325 MB of downloads. Medusa trained its heads on ShareGPT dialogues of the same kind: the heads must learn the distribution of the text the model actually writes

In [ ]:
# Two parquet shards read directly from the Hub. Loading the repo id with data_files would make datasets check
# our split names against the four splits on the dataset card and refuse them.
raw = load_dataset("parquet", data_files={"train": f"hf://datasets/{DATASET}/{TRAIN_FILE}",
                                          "test": f"hf://datasets/{DATASET}/{TEST_FILE}"})
train_rows = raw["train"].shuffle(seed=SEED).select(range(N_TRAIN))
test_rows = raw["test"].shuffle(seed=SEED).select(range(N_TEST))
print(raw)

Each row is a whole conversation. We print the start of one and the distribution of turns per conversation.

In [ ]:
example = train_rows[0]
for turn in example["messages"][:2]:
    print(f"[{turn['role']}] {turn['content'][:300]}\n")
turns = np.array([len(m) for m in train_rows["messages"]])
print(f"turns per conversation: min {turns.min()}, median {np.median(turns):.0f}, max {turns.max()}")

Every conversation is rendered with the model's chat template (role markers and the default system prompt included) before tokenisation, so the heads see text in the exact format the model is served with. Conversations longer than `MAX_LEN` tokens are cut.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL)


def to_tokens(batch):
    """Render each conversation with the chat template, then tokenise and truncate it."""
    texts = [tokenizer.apply_chat_template(m, tokenize=False) for m in batch["messages"]]
    return tokenizer(texts, truncation=True, max_length=MAX_LEN)


train_tok = train_rows.map(to_tokens, batched=True, remove_columns=train_rows.column_names)
test_tok = test_rows.map(to_tokens, batched=True, remove_columns=test_rows.column_names)
lengths = np.array([len(ids) for ids in train_tok["input_ids"]])
print(f"tokens per conversation: median {np.median(lengths):.0f}, "
      f"share truncated at {MAX_LEN}: {(lengths == MAX_LEN).mean():.0%}")
print(f"training tokens per epoch: {lengths.sum():,}")

A `Dataset` returns one conversation as a tensor of token ids. The collate function right-pads the batch and builds `labels`, with padding set to `-100` so the loss ignores it. The labels are the input ids themselves: head $k$ at position $t$ is scored against `labels[t + k + 1]`.

In [ ]:
class ChatDataset(Dataset):
    """Tokenised conversations, one tensor of token ids per item."""

    def __init__(self, tokenised):
        self.ids = tokenised["input_ids"]

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return torch.tensor(self.ids[i])


def pad_collate(sequences):
    """Right-pad to the longest conversation in the batch. Returns input_ids, attention_mask and labels, all (B, T)."""
    input_ids = nn.utils.rnn.pad_sequence(sequences, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_mask = nn.utils.rnn.pad_sequence([torch.ones_like(s) for s in sequences], batch_first=True, padding_value=0)
    labels = input_ids.masked_fill(attention_mask == 0, -100)
    return input_ids, attention_mask, labels


train_loader = DataLoader(ChatDataset(train_tok), batch_size=BATCH_SIZE, shuffle=True, collate_fn=pad_collate,
                          num_workers=NUM_WORKERS, pin_memory=DEVICE == "cuda")
test_loader = DataLoader(ChatDataset(test_tok), batch_size=BATCH_SIZE, collate_fn=pad_collate,
                         num_workers=NUM_WORKERS, pin_memory=DEVICE == "cuda")

input_ids, attention_mask, labels = next(iter(train_loader))
print(input_ids.shape, attention_mask.shape, labels.shape)   # (BATCH_SIZE, T), T <= MAX_LEN

## 2. Model

The target is loaded once, in bfloat16 on an A100, and frozen: no gradient ever reaches it. Its decoder stack `target.model` returns the final hidden state after the last normalisation layer, and `target.lm_head` turns that state into next-token logits. The Medusa heads read the same hidden state.

In [ ]:
target = AutoModelForCausalLM.from_pretrained(TARGET_MODEL, dtype=MODEL_DTYPE).to(DEVICE).eval()
for p in target.parameters():
    p.requires_grad_(False)

D_MODEL = target.config.hidden_size
VOCAB = target.lm_head.weight.shape[0]
n_frozen = sum(p.numel() for p in target.parameters())
print(f"{n_frozen / 1e9:.2f}B frozen parameters | d_model {D_MODEL} | vocabulary {VOCAB:,} | {MODEL_DTYPE}")

A head is a stack of `HEAD_LAYERS` residual blocks followed by a vocabulary projection. The block computes $h + \mathrm{SiLU}(W_1 h)$ with $W_1 = 0$ at initialisation, so it starts as the identity, and the projection is initialised from the frozen language-model head. Before training, head $k$ therefore makes the base model's next-token guess, which is right for position $t + 1$ and usually wrong for $t + k + 1$.

In [ ]:
class ResBlock(nn.Module):
    """h + SiLU(W h): a residual block that starts as the identity because W is zero at initialisation."""

    def __init__(self, d_model):
        super().__init__()
        self.linear = nn.Linear(d_model, d_model)
        nn.init.zeros_(self.linear.weight)
        nn.init.zeros_(self.linear.bias)

    def forward(self, h):                      # (B, T, d_model) -> (B, T, d_model)
        return h + F.silu(self.linear(h))


class MedusaHeads(nn.Module):
    """N_HEADS draft heads on the frozen final hidden state. Head k guesses the token k + 1 positions ahead."""

    def __init__(self, d_model, vocab, n_heads, n_layers, lm_head_weight=None):
        super().__init__()
        self.heads = nn.ModuleList()
        for _ in range(n_heads):
            blocks = [ResBlock(d_model) for _ in range(n_layers)]
            proj = nn.Linear(d_model, vocab, bias=False)         # W_2: (vocab, d_model), one copy per head
            if lm_head_weight is not None:
                proj.weight.data.copy_(lm_head_weight)            # start from the original LM head, as Medusa does
            self.heads.append(nn.Sequential(*blocks, proj))

    def forward(self, hidden):                 # (B, T, d_model) -> list of N_HEADS logits, each (B, T, vocab)
        return [head(hidden) for head in self.heads]


heads = MedusaHeads(D_MODEL, VOCAB, N_HEADS, HEAD_LAYERS, lm_head_weight=target.lm_head.weight.float()).to(DEVICE)
n_params = sum(p.numel() for p in heads.parameters() if p.requires_grad)
print(f"Trainable head parameters: {n_params:,} ({n_params / n_frozen:.0%} of the frozen backbone)")

## 3. Training

Head $k$ at position $t$ is scored with cross-entropy against the token at $t + k + 1$, so its logits lose the last $k + 1$ positions and the labels lose the first $k + 1$. The head losses are summed with weights $\lambda_k = 0.8^k$ and minimised with AdamW under a short linear warm-up and a cosine decay, the schedule of the Medusa training script. The backbone runs once per batch under `no_grad`, and each head's loss is backpropagated on its own, so only one head's logits (about 5 GB in float32 for 8,000 tokens over a 152k vocabulary) is alive at a time.

Expected time: about 6 minutes per epoch on an A100, about 12 minutes for the two epochs plus a short evaluation after each.

In [ ]:
def head_loss(head, k, hidden, labels):
    """Cross-entropy of head k, whose logit at position t is scored against the token at t + k + 1.
    Returns the loss, the number of correct top-1 guesses and the number of scored positions."""
    logits = head(hidden)[:, :-(k + 1)].float()        # (B, T-k-1, vocab): positions that have a target
    targets = labels[:, k + 1:]                        # (B, T-k-1)
    loss = F.cross_entropy(logits.flatten(0, 1), targets.flatten(), ignore_index=-100)
    scored = targets != -100
    correct = ((logits.argmax(-1) == targets) & scored).sum().item()
    return loss, correct, scored.sum().item()


@torch.no_grad()
def final_hidden(input_ids, attention_mask):
    """The frozen target's final hidden state, (B, T, d_model), in the model's dtype."""
    with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
        return target.model(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state

In [ ]:
def train_one_epoch(heads, loader, optimizer, scheduler):
    """One pass over the training set. Returns the mean weighted loss and the top-1 accuracy of each head."""
    heads.train()
    total_loss, correct, seen = 0.0, np.zeros(N_HEADS), np.zeros(N_HEADS)
    # Pass an iterator, not the loader: given a loader, tqdm.auto calls iter() on it once more for
    # itself, which starts an extra set of worker processes every epoch that is never used or shut down.
    progress = tqdm(iter(loader), total=len(loader), desc="train", leave=False)
    for input_ids, attention_mask, labels in progress:
        input_ids, attention_mask, labels = input_ids.to(DEVICE), attention_mask.to(DEVICE), labels.to(DEVICE)
        hidden = final_hidden(input_ids, attention_mask)                 # (B, T, d_model), no graph
        optimizer.zero_grad(set_to_none=True)
        batch_loss = 0.0
        for k, head in enumerate(heads.heads, start=1):
            with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
                loss, c, n = head_loss(head, k, hidden, labels)
            scaler.scale(LOSS_DECAY ** k * loss).backward()              # one head at a time: frees its logits before the next
            batch_loss += LOSS_DECAY ** k * loss.item()
            correct[k - 1] += c
            seen[k - 1] += n
        scaler.step(optimizer)                                           # the scaler is a no-op in bfloat16
        scaler.update()
        scheduler.step()
        total_loss += batch_loss
        progress.set_postfix(loss=f"{batch_loss:.3f}")
    return total_loss / len(loader), correct / seen


@torch.no_grad()
def evaluate(heads, loader):
    """Mean weighted loss and per-head top-1 accuracy on a held-out loader."""
    heads.eval()
    total_loss, correct, seen = 0.0, np.zeros(N_HEADS), np.zeros(N_HEADS)
    for input_ids, attention_mask, labels in loader:
        input_ids, attention_mask, labels = input_ids.to(DEVICE), attention_mask.to(DEVICE), labels.to(DEVICE)
        hidden = final_hidden(input_ids, attention_mask)
        for k, head in enumerate(heads.heads, start=1):
            with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
                loss, c, n = head_loss(head, k, hidden, labels)
            total_loss += LOSS_DECAY ** k * loss.item()
            correct[k - 1] += c
            seen[k - 1] += n
    return total_loss / len(loader), correct / seen

The first evaluation, before any training, is the baseline: heads that still copy the next-token head. Then the epoch loop prints the loss, the accuracy of every head and the elapsed time.

In [ ]:
optimizer = torch.optim.AdamW(heads.parameters(), lr=LR, weight_decay=0.0)
scheduler = get_cosine_schedule_with_warmup(optimizer, WARMUP_STEPS, EPOCHS * len(train_loader))
print(f"{len(train_loader):,} optimizer steps per epoch, {EPOCHS} epochs")

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
val_loss, val_acc = evaluate(heads, test_loader)
history["val_loss"].append(val_loss)
history["val_acc"].append(val_acc)
print("Before training: val loss {:.3f} | head top-1 accuracy {}".format(
    val_loss, " ".join(f"{a:.3f}" for a in val_acc)))

start = time.time()
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(heads, train_loader, optimizer, scheduler)
    val_loss, val_acc = evaluate(heads, test_loader)
    for key, value in zip(history, (train_loss, train_acc, val_loss, val_acc)):
        history[key].append(value)
    print(f"Epoch {epoch}/{EPOCHS}: train loss {train_loss:.3f} | val loss {val_loss:.3f} | "
          f"head top-1 accuracy {' '.join(f'{a:.3f}' for a in val_acc)} | {time.time() - start:.0f}s elapsed")

## 4. Results

The left panel shows the weighted loss, the right one the held-out top-1 accuracy of each head, with epoch 0 as the untrained baseline. The top-1 accuracy of head $k$ is the probability that its guess survives greedy verification once the earlier guesses have, so these curves predict the acceptance length measured below.

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
val_acc = np.array(history["val_acc"])                     # (EPOCHS + 1, N_HEADS)
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(12, 4))
ax_loss.plot(epochs, history["train_loss"], marker="o", label="train")
ax_loss.plot(range(len(history["val_loss"])), history["val_loss"], marker="o", label="validation")
ax_loss.set(xlabel="epoch", ylabel="weighted cross-entropy", title="Loss")
ax_loss.legend()
for k in range(N_HEADS):
    ax_acc.plot(range(len(val_acc)), val_acc[:, k], marker="o", label=f"head {k + 1} (offset {k + 2})")
ax_acc.set(xlabel="epoch (0 = untrained)", ylabel="top-1 accuracy", title="Held-out head accuracy")
ax_acc.legend()
plt.tight_layout()
plt.show()

### The draft-and-verify loop

One step holds the target's committed next token $x_0$ and the hidden state $h$ it came from. Each head reads $h$ and guesses one token, giving the chain $[x_0, \tilde{x}_1, \dots, \tilde{x}_K]$. The target runs once over the chain, with the KV cache of everything before it. Its logits at chain position $i$ say which token it would write after $\tilde{x}_i$, so guess $\tilde{x}_{i+1}$ is accepted if it equals that token, and acceptance stops at the first mismatch. The accepted guesses and $x_0$ are committed, the target's own token after the last accepted position becomes the next $x_0$ for free, the hidden state at that position feeds the heads, and the cache entries of the rejected guesses are cropped away. With no heads the chain is just $[x_0]$ and the same code is plain greedy decoding, which gives a fair baseline.

In [ ]:
@torch.no_grad()
def generate_with_heads(prompt_ids, heads=None, max_new_tokens=MAX_NEW_TOKENS):
    """Greedy decoding with a draft-and-verify step. heads=None gives plain one-token-per-pass decoding.
    Returns the new token ids and, for every target pass, the number of accepted draft tokens."""
    with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
        cache = DynamicCache()
        out = target.model(input_ids=prompt_ids, past_key_values=cache, use_cache=True)
        hidden = out.last_hidden_state[:, -1]                           # (1, d_model), the last prompt position
        next_token = target.lm_head(hidden).argmax(-1)                  # (1,), the target's committed next token
        new_tokens, accepted = [], []
        while len(new_tokens) < max_new_tokens:
            guesses = [head(hidden).argmax(-1) for head in heads.heads] if heads is not None else []
            chain = torch.stack([next_token, *guesses], dim=1)         # (1, K+1): committed token then K guesses
            out = target.model(input_ids=chain, past_key_values=cache, use_cache=True)
            states = out.last_hidden_state                              # (1, K+1, d_model)
            verdict = target.lm_head(states).argmax(-1)                 # (1, K+1): the target's token after each chain token
            matches = verdict[0, :-1] == chain[0, 1:]                   # (K,): guess i is right if it equals the target's token
            n_accepted = int(matches.cumprod(0).sum())                  # accepted prefix: stop at the first mismatch
            committed = chain[0, :n_accepted + 1].tolist()              # the committed token plus the accepted guesses
            new_tokens.extend(committed)
            accepted.append(n_accepted)
            next_token = verdict[:, n_accepted]                         # the target's token after the last accepted one: free
            hidden = states[:, n_accepted]                              # (1, d_model): the heads read this state next
            cache.crop(cache.get_seq_length() - (len(guesses) - n_accepted))   # drop the rejected guesses from the KV cache
            if tokenizer.eos_token_id in committed:
                break
    return new_tokens[:max_new_tokens], accepted

### Acceptance length and speed

Every method decodes the same prompts, the first user messages of held-out conversations, with the same token budget. We record wall-clock time, new tokens per target forward pass (the prompt pass included) and how much of the plain greedy output each method reproduces before its first differing token (the prefix match, measured against the plain run of the same code path, since our loop and `generate` use different kernels). Timing is per prompt with the GPU synchronised, so it includes the Python overhead of the loop, as a user would feel it.

In [ ]:
def chat_prompt(text):
    return tokenizer.apply_chat_template([{"role": "user", "content": text}], tokenize=False, add_generation_prompt=True)


eval_prompts = [chat_prompt(row["prompt"]) for row in test_rows.select(range(N_EVAL_PROMPTS))]


def sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()


def time_method(decode_fn):
    """decode_fn(prompt_ids) -> (new token ids, number of target passes). Runs every prompt and aggregates."""
    outputs, seconds, passes = [], 0.0, 0
    for prompt in eval_prompts:
        ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)
        sync()
        start = time.perf_counter()
        tokens, n_passes = decode_fn(ids)
        sync()
        seconds += time.perf_counter() - start
        outputs.append(tokens)
        passes += n_passes
    n_tokens = sum(len(t) for t in outputs)
    return {"outputs": outputs, "tok_per_s": n_tokens / seconds, "tok_per_pass": n_tokens / passes}


all_accepted = []


def plain_loop(ids):
    tokens, accepted = generate_with_heads(ids, heads=None)
    return tokens, len(accepted) + 1                      # verification passes plus the prompt pass


def medusa_loop(ids):
    tokens, accepted = generate_with_heads(ids, heads)
    all_accepted.extend(accepted)
    return tokens, len(accepted) + 1


heads.eval()
results = {"plain (our loop)": time_method(plain_loop), "Medusa heads (our loop)": time_method(medusa_loop)}
accepted = np.array(all_accepted)
print(f"acceptance length (tokens per target pass): {accepted.mean() + 1:.2f}")
for k in range(1, N_HEADS + 1):
    print(f"  head {k} accepted in {(accepted >= k).mean():.0%} of passes")

### Comparison with a separate draft model

`transformers` implements the classic recipe through `generate(assistant_model=...)`: a smaller model with the same tokenizer drafts tokens greedily and the target verifies them in one pass, accepting up to the first mismatch. We fix the draft length to `N_HEADS` tokens for a like-for-like comparison and count the target's forward passes with a hook.

In [ ]:
draft = AutoModelForCausalLM.from_pretrained(DRAFT_MODEL, dtype=MODEL_DTYPE).to(DEVICE).eval()
draft.generation_config.num_assistant_tokens = N_HEADS
draft.generation_config.num_assistant_tokens_schedule = "constant"     # no adaptive draft length


class PassCounter:
    """Counts the forward passes of a model through a pre-forward hook."""

    def __init__(self, model):
        self.n = 0
        self.handle = model.register_forward_pre_hook(self._count)

    def _count(self, *args):
        self.n += 1


def hf_generate(ids, **kwargs):
    counter = PassCounter(target)
    out = target.generate(ids, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                          pad_token_id=tokenizer.pad_token_id, **kwargs)
    counter.handle.remove()
    return out[0, ids.shape[1]:].tolist(), counter.n                    # all target passes, the prompt pass included


results["plain (generate)"] = time_method(hf_generate)
results["assisted generate, 0.5B draft"] = time_method(lambda ids: hf_generate(ids, assistant_model=draft))

baseline = {"our loop": results["plain (our loop)"]["tok_per_s"], "generate": results["plain (generate)"]["tok_per_s"]}
def prefix_match(a, b):
    """Share of the reference output reproduced before the first differing token."""
    n = next((i for i, (x, y) in enumerate(zip(a, b)) if x != y), min(len(a), len(b)))
    return n / max(len(b), 1)


references = {"our loop": results["plain (our loop)"]["outputs"], "generate": results["plain (generate)"]["outputs"]}
print(f"{'method':<32} {'tokens/s':>9} {'tok/pass':>9} {'speedup':>8} {'prefix match':>13}")
for name, r in results.items():
    family = "our loop" if "our loop" in name else "generate"
    match = np.mean([prefix_match(a, b) for a, b in zip(r["outputs"], references[family])])
    print(f"{name:<32} {r['tok_per_s']:>9.1f} {r['tok_per_pass']:>9.2f} {r['tok_per_s'] / baseline[family]:>8.2f} {match:>13.0%}")

i = 0
print("\nPrompt:", test_rows[i]["prompt"][:200])
print("\nMedusa output:", tokenizer.decode(results["Medusa heads (our loop)"]["outputs"][i], skip_special_tokens=True)[:600])

**What to notice**

Before training every head copies the next-token head, so acceptance is rare. After two epochs the first head is right far more often than the third: guessing further ahead without seeing the tokens in between is harder, which is why the loss weights decay with $k$ and why the acceptance length stays well below $K + 1$. The speedup is smaller than the acceptance length, because every pass also runs the heads and verifies $K + 1$ tokens, and the Python loop adds a fixed cost per pass. The assisted-generation baseline reaches a comparable acceptance length but pays for a separate 0.5B model running $K$ serial steps per pass, while the heads cost a few matrix products on a state that is already there. In exact arithmetic greedy verification reproduces plain decoding token for token. In bfloat16 the logits of a token verified inside a chunk of $K + 1$ and of the same token decoded alone differ slightly, so at a position where the top two tokens are nearly tied the two runs can part ways, and from there on the texts differ: the prefix match shows how far the agreement goes, not an error in the method.

The heads are the only new weights, so we save just their state dict (about 2.8 GB in float32), rebuild the module without the language-model-head initialisation, load the weights and check the held-out accuracy.

In [ ]:
torch.save(heads.state_dict(), "medusa_heads.pt")

reloaded = MedusaHeads(D_MODEL, VOCAB, N_HEADS, HEAD_LAYERS).to(DEVICE)
reloaded.load_state_dict(torch.load("medusa_heads.pt", map_location=DEVICE))
val_loss, val_acc = evaluate(reloaded, test_loader)
print("Reloaded heads: val loss {:.3f} | head top-1 accuracy {}".format(val_loss, " ".join(f"{a:.3f}" for a in val_acc)))

## Summary

- Speculative decoding turns idle GPU compute into tokens: a cheap draft proposes several tokens and the target verifies them in one pass, and greedy verification leaves the output unchanged.
- Medusa heads are a draft that lives inside the target: small residual blocks on the final hidden state, initialised from the language-model head and trained with a per-offset cross-entropy while the backbone stays frozen.
- The acceptance length is set by the heads' offset accuracy, which falls with the distance guessed ahead, and the wall-clock gain is the acceptance length minus the cost of drafting and verifying.
- A separate draft model gives a comparable acceptance length but must be found, aligned with the target's tokenizer and served, which is the problem Medusa removes.

## Exercises

1. **More heads.** Set `N_HEADS = 5` and retrain. Compare the acceptance length and the speedup with three heads, and look at how often heads 4 and 5 are accepted.
2. **Loss weighting.** Set `LOSS_DECAY = 1.0` so every head counts equally. Compare the per-head accuracies and the acceptance length with the decayed weighting.
3. **Only the assistant's words.** Medusa masks the user turns so the heads learn only what the model writes. Build the labels with `tokenizer.apply_chat_template(..., return_assistant_tokens_mask=True)` or by locating the assistant spans, set the other labels to `-100`, and compare the acceptance length on the same prompts.
4. **Draft length.** In the assisted-generation comparison, set `num_assistant_tokens` to 2 and to 8 and switch the schedule to `"heuristic"`. Compare tokens per pass and tokens per second with the fixed length of 3.
5. **Batch size.** Time the target's forward over `N_HEADS + 1` tokens against one token at batch sizes 1, 8 and 32 with a cached prefix. Explain from the ratio why the speedup of this lab belongs to small batches.

## Further Reading

- [Leviathan et al., 2023. Fast Inference from Transformers via Speculative Decoding](https://arxiv.org/abs/2211.17192)
- [Chen et al., 2023. Accelerating Large Language Model Decoding with Speculative Sampling](https://arxiv.org/abs/2302.01318)
- [Cai et al., 2024. Medusa: Simple LLM Inference Acceleration Framework with Multiple Decoding Heads](https://arxiv.org/abs/2401.10774)
- [Ankner et al., 2024. Hydra: Sequentially-Dependent Draft Heads for Medusa Decoding](https://arxiv.org/abs/2402.05109)
- [Li et al., 2025. EAGLE-3: Scaling up Inference Acceleration of Large Language Models via Training-Time Test](https://arxiv.org/abs/2503.01840)
- [Gante, 2023. Assisted Generation: a new direction toward low-latency text generation](https://huggingface.co/blog/assisted-generation)

### Contributed by: Mohamed Eltayeb